In [ ]:
#!/usr/bin/env python3
from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import torch
import json
from huggingface_hub import login
import imageio
import trimesh


from trellis2.utils.vae_helpers import *
# from trellis2.utils.offline_render import *

# Helper for Python < 3.10 compatibility
class nullcontext:
    def __enter__(self):
        return None

    def __exit__(self, *exc):
        return False

# ---------------------------------------------------------------------
# Environment tweaks
# ---------------------------------------------------------------------
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["ATTN_BACKEND"] = "xformers"

# Check if we are in a notebook
try:
    from trellis2.pipelines import Trellis2ImageTo3DPipeline
    from trellis2.modules.sparse import SparseTensor
    import trellis2.models as models
except ImportError:
    print(
        "Error: trellis2 module not found. Make sure you are in the correct environment."
    )

from PIL import Image

os.environ["PYOPENGL_PLATFORM"] = (
    "egl"  # must come before importing pyrender, OpenGL, etc.
)

# --- pyrender imports must come after pyglet option set ---
import pyglet

pyglet.options["shadow_window"] = False


from trellis2.utils import render_utils

In [ ]:
class Config:
    # Authentication
    hf_token: str = "YOUR_HUGGINGFACE_TOKEN_HERE"

    # Paths
    dataset_root: Path = Path("datasets/AutoBrep_Dataset")
    out_dir: Path = Path(
        "datasets/AutoBrep_Dataset/decode_shapes/exp3"
    )  # Output directory

    # Model Settings
    model_id: str = "microsoft/TRELLIS.2-4B"
    decoder_pretrained: list[str] = [
        "microsoft/TRELLIS-image-large/ckpts/ss_dec_conv3d_16l8_fp16",
        "microsoft/TRELLIS.2-4B/ckpts/shape_dec_next_dc_f16c32_fp16",
    ]
    low_vram: bool = False  # Enable low VRAM mode
    dtype: str = "fp32"  # Choices: "fp16", "bf16", "fp32"

    # Decoding Logic
    use_ss_decoder: bool = False  # Use sparse_structure_decoder for coords
    ss_target_res: int = 64  # Target resolution for SS decoder
    pool_on_cpu: bool = True  # Pool SS occupancy on CPU to save VRAM

    # Processing
    decode_resolutions: list[int] = [1024, 512, 256]
    decimate_faces: int = 16_777_216  # If >0, simplify mesh


args = Config()

# Authenticate
if args.hf_token and args.hf_token != "YOUR_HUGGINGFACE_TOKEN_HERE":
    login(token=args.hf_token)
else:
    print(
        "Warning: No HF token provided. Ensure you are logged in via CLI or provide token in Config."
    )

# Create output dir
args.out_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory set to: {args.out_dir}")

In [ ]:
def generate_combinations_v1(
    ss_dir: Path, shape_dir: Path
) -> Iterator[Tuple[str, Path, Path]]:
    ss_map: Dict[str, Path] = {p.stem: p for p in ss_dir.glob("*.npz")}
    shape_map: Dict[str, Path] = {p.stem: p for p in shape_dir.glob("*.npz")}

    with open("datasets/AutoBrep_Dataset/vae_exps.json") as f:
        d = json.load(f)
        for item in d:
            name = item[0]
            ss_map_name = item[1]
            shape_map_name = item[2]
            if ss_map_name in ss_map and shape_map_name in shape_map:
                yield name, ss_map[ss_map_name], shape_map[shape_map_name], ss_map[
                    shape_map_name
                ], shape_map[ss_map_name]


def generate_combinations_v2(
    ss_dir: Path, shape_dir: Path
) -> Iterator[Tuple[str, Path, Path]]:
    ss_map: Dict[str, Path] = {p.stem: p for p in ss_dir.glob("*.npz")}
    shape_map: Dict[str, Path] = {p.stem: p for p in shape_dir.glob("*.npz")}

    with open("o-voxel/examples/parquet_name2sha.json", "r") as file:
        parquet_name2sha = json.load(file)

    for key in parquet_name2sha.keys():
        if parquet_name2sha[key] in shape_map:
            yield key, shape_map[parquet_name2sha[key]], ss_map[parquet_name2sha[key]] if parquet_name2sha[key] in ss_map else None

In [ ]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("This script expects a CUDA GPU.")

dtype_map = {"fp16": torch.float16, "bf16": torch.bfloat16, "fp32": torch.float32}
compute_dtype = dtype_map[args.dtype]

LOGGER.info(f"Loading pipeline {args.model_id} ...")

# # Load Pipeline
# pipeline = Trellis2ImageTo3DPipeline.from_pretrained(args.model_id)
# pipeline.cuda()
# pipeline.low_vram = bool(args.low_vram)

decoder_slat = (
    models.from_pretrained(args.decoder_pretrained[0]).eval().cuda()
)
decoder_slat.low_vram = bool(args.low_vram)

decoder_shape = models.from_pretrained(args.decoder_pretrained[1]).eval().cuda()
decoder_shape.low_vram = bool(args.low_vram)

torch.cuda.reset_peak_memory_stats()
log_cuda_memory("after_pipeline_load")

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [17]:
# Create context for mixed precision
autocast_ctx = (
    torch.autocast(device_type="cuda", dtype=compute_dtype)
    if compute_dtype != torch.float32
    else nullcontext()
)

# Run Inference
with torch.inference_mode(), autocast_ctx:
    for res in [1024, 512, 256]:
        # Construct paths based on resolution
        ss_dir = (
            args.dataset_root / "ss_latents" / f"ss_enc_conv3d_16l8_fp16_64_{res}"
        )
        shape_dir = (
            args.dataset_root
            / "shape_latents"
            / f"shape_enc_next_dc_f16c32_fp16_{res}"
        )

        if not ss_dir.exists() or not shape_dir.exists():
            LOGGER.warning(
                f"Missing dirs for res={res}: \n  {ss_dir} \n  {shape_dir}"
            )
            continue

        LOGGER.info(
            f"Decoding resolution={res} from:\n  ss: {ss_dir}\n  shape: {shape_dir}"
        )

        # Process files
        exp_combinations = generate_combinations_v2(ss_dir, shape_dir)

        for (
            key,
            shape_path,
            ss_path,
        ) in exp_combinations:
            torch.cuda.reset_peak_memory_stats()

            LOGGER.info(f"Processing: {key}")

            # 1. Load latents (CPU)
            feats_np, coords_np = load_shape_latent_npz(shape_path)
            z_np = load_ss_latent_npz(ss_path) if args.use_ss_decoder else None

            # 2. Move to GPU
            feats = torch.from_numpy(feats_np).to(
                device=device, dtype=compute_dtype, non_blocking=True
            )

            enc_coords = torch.from_numpy(coords_np).to(
                device=device, dtype=torch.int32, non_blocking=True
            )
            enc_coords = add_batch_dim_if_missing(enc_coords)

            LOGGER.info(
                f"Loaded latents: feats shape {feats.shape}, coords shape {enc_coords.shape}"
            )

            # 3. Optional: Decode Sparse Structure coords
            if args.use_ss_decoder:
                LOGGER.info("Decoding sparse structure coordinates...")
                z_s = torch.from_numpy(z_np).to(
                    device=device, dtype=compute_dtype, non_blocking=True
                )
                LOGGER.info(f"Loaded latents: z_s shape {z_s.shape} ")
                if z_s.ndim == 4:
                    z_s = z_s.unsqueeze(0)

                dec_coords, decoded = decode_sparse_structure_coords(
                    decoder_slat,
                    z_s,
                    target_resolution=64,
                    pool_on_cpu=args.pool_on_cpu,
                )
                dec_coords = dec_coords.to(device)

                LOGGER.info(f"Decoded latents coords shape {dec_coords.shape}")
                coords = dec_coords
            else:
                coords = enc_coords

            ## take intersection of dec_coords and enc_coords
            # dec_coords_set = set(map(tuple, dec_coords.cpu().numpy()))
            # enc_coords_set = set(map(tuple, enc_coords.cpu().numpy()))
            # coords_torch = (
            #     torch.from_numpy(np.array(list(dec_coords_set & enc_coords_set)))
            #     .to(torch.int32)
            #     .contiguous()
            #     .to(device)
            # )
            ## update feats to match coords
            # coord_to_index = {tuple(c.cpu().numpy()): i for i, c in enumerate(dec_coords)}
            # indices = [coord_to_index[tuple(c.cpu().numpy())] for c in coords_torch]
            # feats_temp = feats_temp[indices]
            # feats_temp =  feats[indices]  # feats_temp

            # for k in [10, 50, 100, 250, 500, 1500, 3000]:
            # nn_idx, nn_dist = nearest_neighbor_kdtree(coords_torch, coords_torch, k=k)
            # coords = coords_torch[nn_idx[:1]][0]
            # feats = feats_temp[nn_idx[:1]][0]

            # ## NN features
            # alpha = 0.0
            # nn_idx, nn_dist = nearest_neighbor_kdtree(dec_coords, enc_coords)
            # feats = feats[nn_idx] * alpha + feats_temp * (1.0 - alpha)
            # coords = dec_coords

            # LOGGER.info(
            #     f"After intersection, feats shape: {feats.shape}, coords shape: {coords.shape}"
            # )

            # export_voxels_as_cubes_mesh(
            #     args.out_dir
            #     / f"VOXELS_{key}_{res}{'_from_dec-coords.ply' if args.use_ss_decoder else '_from_enc-coords.ply'}",
            #     coords,
            #     voxel_size=1.0,
            # )

            # 4. Construct SparseTensor
            shape_slat = SparseTensor(feats=feats, coords=coords)
            LOGGER.info(
                f"Constructing SparseTensor with feats {feats.shape} and coords {coords.shape}"
            )

            log_cuda_memory(f"{key}: before_decode")

            # 5. Decode Meshes
            meshes = decode_meshes_from_shape_slat(
                decoder_shape,
                shape_slat,
                resolution=res,
                return_subs=False,
            )

            # 6. Post-processing
            mesh = meshes[0]
            # mesh.fill_holes()

            if args.decimate_faces and args.decimate_faces > 0:
                mesh.simplify(args.decimate_faces)

            # 7. Export
            v = mesh.vertices.detach().cpu().numpy().astype(np.float32)
            f = mesh.faces.detach().cpu().numpy().astype(np.int32)

            out_name = f"{key}_{res}" + (
                "_from_dec-coords" if args.use_ss_decoder else "_from_enc-coords"
            )
            out_path = args.out_dir / f"{out_name}"

            # write_ply_binary(f"{out_path}.ply", v, f)

            trellis_video = render_utils.make_vis_frames(
                render_utils.render_video(
                    mesh, num_frames=70
                )
            )
            imageio.mimsave(
                f"{out_path}.mp4",
                trellis_video,
                fps=15,
            )

            log_cuda_memory(f"{key}: after_export")
            LOGGER.info(f"Wrote {out_path}")

            # 8. Cleanup to prevent VRAM accumulation
            cleanup_cuda(shape_slat, feats, coords, meshes, mesh)
            if args.use_ss_decoder:
                cleanup_cuda(z_s)


LOGGER.info("Done.")

2026-04-10 16:42:08,320 - INFO - Decoding resolution=1024 from:
  ss: datasets/AutoBrep_Dataset/ss_latents/ss_enc_conv3d_16l8_fp16_64_1024
  shape: datasets/AutoBrep_Dataset/shape_latents/shape_enc_next_dc_f16c32_fp16_1024
2026-04-10 16:42:08,331 - INFO - Processing: sample_000037_edges
2026-04-10 16:42:08,362 - INFO - Loaded latents: feats shape torch.Size([2606, 32]), coords shape torch.Size([2606, 4])
2026-04-10 16:42:08,365 - INFO - Constructing SparseTensor with feats torch.Size([2606, 32]) and coords torch.Size([2606, 4])
2026-04-10 16:42:08,367 - INFO - [sample_000037_edges: before_decode] CUDA alloc=1.08GB reserved=2.66GB peak=1.08GB


Rendering: 100%|██████████| 70/70 [00:04<00:00, 16.87it/s]


2026-04-10 16:42:15,332 - INFO - [sample_000037_edges: after_export] CUDA alloc=1.04GB reserved=2.67GB peak=1.54GB
2026-04-10 16:42:15,334 - INFO - Wrote datasets/AutoBrep_Dataset/decode_shapes/exp3/sample_000037_edges_1024_from_enc-coords
2026-04-10 16:42:15,765 - INFO - Processing: sample_000017_edges
2026-04-10 16:42:15,773 - INFO - Loaded latents: feats shape torch.Size([4040, 32]), coords shape torch.Size([4040, 4])
2026-04-10 16:42:15,774 - INFO - Constructing SparseTensor with feats torch.Size([4040, 32]) and coords torch.Size([4040, 4])
2026-04-10 16:42:15,776 - INFO - [sample_000017_edges: before_decode] CUDA alloc=1.04GB reserved=1.06GB peak=1.04GB


Rendering: 100%|██████████| 70/70 [00:04<00:00, 16.86it/s]


2026-04-10 16:42:22,989 - INFO - [sample_000017_edges: after_export] CUDA alloc=1.05GB reserved=1.77GB peak=1.65GB
2026-04-10 16:42:22,991 - INFO - Wrote datasets/AutoBrep_Dataset/decode_shapes/exp3/sample_000017_edges_1024_from_enc-coords
2026-04-10 16:42:23,375 - INFO - Processing: sample_000098_surface
2026-04-10 16:42:23,397 - INFO - Loaded latents: feats shape torch.Size([14148, 32]), coords shape torch.Size([14148, 4])
2026-04-10 16:42:23,398 - INFO - Constructing SparseTensor with feats torch.Size([14148, 32]) and coords torch.Size([14148, 4])
2026-04-10 16:42:23,400 - INFO - [sample_000098_surface: before_decode] CUDA alloc=1.05GB reserved=1.08GB peak=1.05GB


Rendering:  83%|████████▎ | 58/70 [00:06<00:01,  8.52it/s]


KeyboardInterrupt: 

## Visualizations

In [ ]:
# viz_mesh = trimesh.Trimesh(vertices=v, faces=f)
# viz_mesh.apply_scale(0.6)
# center = viz_mesh.bounds.mean(axis=0)  # (min+max)/2
# viz_mesh.apply_translation(-center)
# # mesh.rezero()
# # mesh.fix_normals()
# viz_mesh = Mesh.from_trimesh(viz_mesh, smooth=True)

In [ ]:
# base_pose = np.array(
#     [
#         [1.0, 0.0, 0.0, 0.0],
#         [0.0, 0.0, -1.0, 0.0],
#         [0.0, 1.0, 0.0, -0.1],
#         [0.0, 0.0, 0.0, 1.0],
#     ]
# )
# direc_l = DirectionalLight(color=np.ones(3), intensity=1.0)
# spot_l = SpotLight(
#     color=np.ones(3),
#     intensity=1.0,
#     innerConeAngle=np.pi / 16,
#     outerConeAngle=np.pi / 6,
# )
# cam = PerspectiveCamera(yfov=(np.pi / 4.0))
# cam_pose = np.array(
#     [
#         #  right_x  up_x   localZ_x   cam_x
#         [0.0, 0.0, 1.0, 0.8],
#         #  right_y  up_y   localZ_y   cam_y
#         [1.0, 0.0, 0.0, 0.0],
#         #  right_z  up_z   localZ_z   cam_z
#         [0.0, 1.0, 0.0, -0.1],
#         [0.0, 0.0, 0.0, 1.0],
#     ]
# )
# scene = Scene(
#     ambient_light=np.array([0.8, 0.3, 0.88, 1.0]), bg_color=[1.0, 1.0, 1.0, 1.0]
# )
# node = scene.add(viz_mesh, pose=base_pose)
# _ = scene.add(direc_l, pose=cam_pose)
# # _ = scene.add(spot_l, pose=cam_pose)
# cam_node = scene.add(cam, pose=cam_pose)
# # ==============================================================================
# # Offscreen renderer
# # ==============================================================================
# width, height = 512 * 2, 512 * 2
# r = OffscreenRenderer(viewport_width=width, viewport_height=height)
# # ==============================================================================
# # Turntable loop
# # ==============================================================================
# # Number of frames around the full 360° (you can increase for smoother animation)
# num_frames = 3
# # Compute object center in world coords (here we use the translation part of base_pose)
# object_center = base_pose[:3, 3]
# axis_vector = [0, 0.6, 1]
# combined = []
# for i, angle in enumerate(np.linspace(0, 2 * np.pi, num_frames, endpoint=False)):
#     # Build a rotation matrix around Y axis centered on object_center
#     R = trimesh.transformations.rotation_matrix(
#         angle, axis_vector, point=object_center
#     )
#     # Apply rotation to the base pose
#     new_pose = R.dot(base_pose)
#     # Update the node’s transform
#     node.matrix = new_pose
#     # Render
#     color, depth = r.render(scene, flags=pyrc.RenderFlags.SKIP_CULL_FACES)  #
#     combined.append(color)
# combined = np.concatenate(combined, axis=1)
# Image.fromarray(combined, mode="RGB")
# r.delete()

In [ ]:
# plt.figure(figsize=(20, 12))
# plt.imshow(combined)
# plt.axis("off")
# plt.show()

In [ ]:
# plot_voxels_scatter(coords, voxel_size=1, origin=(0, 0, 0))
# plot_voxel_slice(coords, axis="z", index=None)

In [ ]:
# from matplotlib import animation
# from IPython.display import HTML

# RESHAPE = False
# if RESHAPE:
#     ratio = decoded.shape[2] // 16
#     decoded = decoded.to("cpu")
#     pooled = F.max_pool3d(decoded.to(torch.float16), ratio, ratio, 0) > 0.5
#     cube1 = pooled[0, 0].numpy().astype(bool)
# else:
#     cube1 = decoded[0, 0].cpu().numpy().astype(bool)

# # combine objects (already boolean)
# voxelarray = cube1

# # set colors
# colors = np.empty(voxelarray.shape, dtype=object)
# colors[voxelarray] = "blue"

# # --- build figure ---
# fig = plt.figure(figsize=(8, 8))
# ax = fig.add_subplot(111, projection="3d")

# # draw voxels once
# vox = ax.voxels(voxelarray, facecolors=colors, edgecolor="k")

# # nicer view limits/aspect (optional)
# ax.set_xlim(0, voxelarray.shape[0])
# ax.set_ylim(0, voxelarray.shape[1])
# ax.set_zlim(0, voxelarray.shape[2])
# ax.grid(False)
# ax.set_xticks([])
# ax.set_yticks([])
# ax.set_zticks([])


# # rotate camera
# def update(frame):
#     ax.view_init(elev=10, azim=frame)  # azim rotates around
#     return []


# anim = animation.FuncAnimation(
#     fig,
#     update,
#     frames=np.linspace(0, 360, 12),  # 120 frames for a full turn
#     interval=50,  # ms between frames
#     blit=False,
# )

# plt.close(fig)  # prevents a static figure from showing above the animation
# # HTML(anim.to_jshtml())
# anim.save(
#     args.out_dir / f"{key}_{res}{'_sscoords.mp4' if args.use_ss_decoder else '.mp4'}",
#     writer="ffmpeg",
#     fps=2,
#     dpi=150,
# )